In [1]:
from pyspark.sql import SparkSession

# Cấu hình
spark = SparkSession.builder \
    .appName("Iceberg-Fix-Final") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "rest") \
    .config("spark.sql.catalog.my_catalog.uri", "http://rest:8181") \
    .config("spark.sql.catalog.my_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.my_catalog.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.my_catalog.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "mypassword") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()


26/05/05 05:30:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS my_catalog.bronze")

DataFrame[]

In [ ]:
# Cách 1 Đọc trực tiếp CSV (không khuyến khích)

# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col, sum as _sum


# df = spark.read.csv("/home/iceberg/my_csv/transactions_data.csv", header=True, inferSchema=True)

# df.show(10)
# df.printSchema()

# clean_df = df.dropna(subset=["store_id", "product_id", "quantity", "price"]) \
#             .filter(col("quantity") > 0) \
#             .filter(col("price") > 0)

# clean_df = clean_df.withColumn("revenue", col("quantity") * col("price"))

# clean_df.select(_sum("revenue")).show()

# clean_df.groupBy("store_id").sum("revenue").show()

# clean_df.groupBy("category").sum("revenue").show()

# clean_df.groupBy("product_id") \
#        .sum("revenue") \
#        .orderBy(col("sum(revenue)").desc()) \
#        .show(10)



In [3]:
# Cách 2 Chuyển CSV về Parquet rồi đọc Parquet

# 1. Đọc CSV bằng Spark
df_transactions = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/home/iceberg/my_csv/transactions_data.csv")

# 2. Kiểm tra schema (Spark sẽ tự hiểu quantity là int, date là date nhờ inferSchema)
df_transactions.printSchema()

# 3. Ghi vào Lakehouse (Lớp Bronze)
# Chúng ta sẽ dùng tính năng Partition của Iceberg để chia dữ liệu theo ngày giao dịch
df_transactions.writeTo("my_catalog.bronze.transactions") \
    .tableProperty("format-version", "2") \
    .partitionedBy("transaction_date") \
    .createOrReplace()

root
 |-- transaction_id: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- transaction_date: date (nullable = true)



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum

# 1. Đọc dữ liệu từ Iceberg Table (Thay vì đọc file CSV thô)
# Spark sẽ tự động đọc các file Parquet bên dưới MinIO dựa trên Metadata
df = spark.read.table("my_catalog.bronze.transactions")

# 2. Kiểm tra dữ liệu
df.show(10)
df.printSchema()

# 3. Làm sạch dữ liệu (Data Cleaning)
# Loại bỏ các dòng lỗi và tính toán revenue
clean_df = df.dropna(subset=["store_id", "product_id", "quantity", "price"]) \
             .filter((col("quantity") > 0) & (col("price") > 0))

clean_df = clean_df.withColumn("revenue", col("quantity") * col("price"))

# 4. Phân tích dữ liệu (Aggregation)
print("--- Tổng doanh thu toàn hệ thống ---")
clean_df.select(_sum("revenue").alias("total_revenue")).show()

print("--- Doanh thu theo cửa hàng ---")
clean_df.groupBy("store_id") \
        .agg(_sum("revenue").alias("store_revenue")) \
        .show()

print("--- Doanh thu theo danh mục sản phẩm ---")
clean_df.groupBy("category") \
        .agg(_sum("revenue").alias("category_revenue")) \
        .show()

print("--- Top 10 sản phẩm doanh thu cao nhất ---")
clean_df.groupBy("product_id") \
        .agg(_sum("revenue").alias("product_revenue")) \
        .orderBy(col("product_revenue").desc()) \
        .show(10)

In [8]:
# Việc chia tầng Bronze (Thô/CSV) -> Silver (Sạch/Parquet) là quy trình chuẩn của các kỹ sư dữ liệu tại các tập đoàn lớn.

clean_df.writeTo("my_catalog.silver.transactions_clean") \
    .tableProperty("format-version", "2") \
    .partitionedBy("category") \
    .createOrReplace()